# Estruturas encadeadas

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Montar uma lista ligada com nós alocados com malloc, cada um apontando para o próximo, e inserir, remover e liberar nós com segurança.
- Usar listas ligadas como pilhas e filas, e listas duplamente ligadas para andar nos dois sentidos.
- Guardar e achar valores numa árvore binária de busca, e imprimi-los em ordem com um percurso recursivo.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Um nó aponta para o próximo
2. Percorrendo a lista
3. Montando uma lista com malloc
4. Liberando uma lista
5. Pondo no fim
6. Tirando um nó
7. Além das listas simples

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## 1. Um nó aponta para o próximo

Uma **lista ligada** é uma corrente de **nós**. Cada nó guarda um valor e um ponteiro para o próximo nó; o último aponta para `NULL`. A struct se refere a si mesma, então precisa de um nome antes de o `typedef` terminar:

```c
typedef struct Node {
    int value;
    struct Node *next;
} Node;
```

Abaixo, três nós são criados como variáveis locais comuns e ligados à mão.

In [ ]:
#include <stdio.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    printf("%d -> %d -> %d\n", a.value, a.next->value, a.next->next->value);
    printf("depois de c vem %s\n", c.next == NULL ? "NULL" : "outro nó");
    return 0;
}

🧪 **Experimente:** insira um nó `x` com valor 6 entre `a` e `b` mudando só ponteiros: `x.next = &b; a.next = &x;`.

## 2. Percorrendo a lista

Para visitar cada nó, comece um ponteiro na cabeça (head) e siga o `next` até ele virar `NULL`:

```c
for (const Node *p = head; p != NULL; p = p->next) { … }
```

Não há índice: chegar ao 100º nó quer dizer seguir 100 ponteiros.

In [ ]:
#include <stdio.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

void print_list(const Node *head) {
    for (const Node *p = head; p != NULL; p = p->next) {
        printf("%d -> ", p->value);
    }
    printf("NULL\n");
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    print_list(&a);
    print_list(&b);
    print_list(NULL);
    return 0;
}

🧪 **Experimente:** imprima só os valores maiores que 4.

### ✏️ Exercício: O tamanho de uma lista

Conte os nós seguindo `next` até NULL. O teste monta 7 → 5 → 3 com nós na stack, então nada precisa ser liberado.

In [ ]:
typedef struct Node {
    int value;
    struct Node *next;
} Node;

int list_length(const Node *head) {
    // TODO: ande com um ponteiro até NULL
    return 0;
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    check_int("três nós", list_length(&a), 3);
    check_int("a partir do meio", list_length(&b), 2);
    check_int("vazia", list_length(NULL), 0);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
typedef struct Node {
    int value;
    struct Node *next;
} Node;

int list_length(const Node *head) {
    int n = 0;
    for (const Node *p = head; p != NULL; p = p->next) {
        n++;
    }
    return n;
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    check_int("três nós", list_length(&a), 3);
    check_int("a partir do meio", list_length(&b), 2);
    check_int("vazia", list_length(NULL), 0);
    return 0;
}
```

</details>

### ✏️ Exercício: A soma de uma lista

Some os valores de todos os nós.

In [ ]:
typedef struct Node {
    int value;
    struct Node *next;
} Node;

int list_sum(const Node *head) {
    // TODO: percorra a lista somando p->value
    return 0;
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    check_int("7 + 5 + 3", list_sum(&a), 15);
    check_int("vazia", list_sum(NULL), 0);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
typedef struct Node {
    int value;
    struct Node *next;
} Node;

int list_sum(const Node *head) {
    int sum = 0;
    for (const Node *p = head; p != NULL; p = p->next) {
        sum += p->value;
    }
    return sum;
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    check_int("7 + 5 + 3", list_sum(&a), 15);
    check_int("vazia", list_sum(NULL), 0);
    return 0;
}
```

</details>

## 3. Montando uma lista com malloc

Listas de verdade crescem enquanto o programa roda, então cada nó vem do `malloc`. Pôr na **cabeça** é o mais simples: o nó novo aponta para a cabeça antiga e vira a nova cabeça. Isso leva o mesmo tempo, seja qual for o tamanho da lista. Os slides mostram a lista crescendo e sendo liberada.

#### 🎞️ Montando e liberando uma lista ligada

![Empilhar 3, 5 e 7 na head dá 7 → 5 → 3 → NULL; para liberar, guarde head->next antes de liberar cada nó](linked-list-steps.pt-br.svg)

In [ ]:
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *push(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) {
        exit(1);
    }
    n->value = value;
    n->next = head;      // o nó novo aponta para a cabeça antiga
    return n;            // e vira a nova cabeça
}

int main(void) {
    Node *head = NULL;
    for (int v = 3; v <= 7; v += 2) {
        head = push(head, v);
    }
    for (Node *p = head; p != NULL; p = p->next) {
        printf("%d ", p->value);
    }
    printf("\n");
    while (head != NULL) {           // libera cada nó
        Node *next = head->next;
        free(head);
        head = next;
    }
    return 0;
}

🧪 **Experimente:** por que a lista sai na ordem contrária dos pushes? Depois esqueça de guardar o resultado: `push(head, v);` sozinho. O que a lista tem agora?

## 4. Liberando uma lista

Todo nó precisa ser liberado, um por um. A ordem importa: **guarde `head->next` antes de `free(head)`**, porque depois do free o nó não pode mais ser lido, `next` incluído.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *push(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) {
        exit(1);
    }
    n->value = value;
    n->next = head;
    return n;
}

int free_list(Node *head) {
    int freed = 0;
    while (head != NULL) {
        Node *next = head->next;   // guarde antes
        free(head);
        head = next;
        freed++;
    }
    return freed;
}

int main(void) {
    Node *head = NULL;
    for (int i = 0; i < 5; i++) {
        head = push(head, i);
    }
    printf("liberou %d nós\n", free_list(head));
    return 0;
}

🧪 **Experimente:** mude o laço para `free(head); head = head->next;` e rode: a verificação de memória acusa a leitura depois do free.

## 5. Pondo no fim

Para pôr no **fim**, ande até o último nó (aquele cujo `next` é `NULL`) e ligue o nó novo ali. Andar deixa isso mais lento em listas longas; guardar um ponteiro `tail` para o último nó evita a caminhada. Uma lista vazia é um caso especial: o nó novo vira a cabeça.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *append(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) {
        exit(1);
    }
    n->value = value;
    n->next = NULL;
    if (head == NULL) {
        return n;                   // a lista estava vazia
    }
    Node *last = head;
    while (last->next != NULL) {
        last = last->next;
    }
    last->next = n;
    return head;
}

int main(void) {
    Node *head = NULL;
    for (int v = 1; v <= 4; v++) {
        head = append(head, v * 10);
    }
    for (Node *p = head; p != NULL; p = p->next) {
        printf("%d ", p->value);
    }
    printf("\n");
    while (head != NULL) {
        Node *next = head->next;
        free(head);
        head = next;
    }
    return 0;
}

🧪 **Experimente:** compare com o passo do push: agora a ordem é a das chamadas. Tire o caso `if (head == NULL)` e rode: o que dá errado com o primeiro valor?

## 6. Tirando um nó

Para tirar um nó, faça o nó **anterior** apontar para o nó **seguinte** e depois libere-o. Guarde um ponteiro `prev` enquanto anda. Tirar a cabeça é o caso especial: a cabeça passa para o segundo nó.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *push(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) {
        exit(1);
    }
    n->value = value;
    n->next = head;
    return n;
}

Node *remove_value(Node *head, int value) {
    Node *prev = NULL;
    for (Node *p = head; p != NULL; prev = p, p = p->next) {
        if (p->value == value) {
            if (prev == NULL) {
                head = p->next;        // tirando a cabeça
            } else {
                prev->next = p->next;  // pula o p
            }
            free(p);
            return head;
        }
    }
    return head;                       // não achou: nada muda
}

void print_list(const Node *head) {
    for (const Node *p = head; p != NULL; p = p->next) {
        printf("%d ", p->value);
    }
    printf("\n");
}

int main(void) {
    Node *head = NULL;
    for (int v = 1; v <= 5; v++) {
        head = push(head, v);
    }
    print_list(head);
    head = remove_value(head, 3);
    head = remove_value(head, 5);
    head = remove_value(head, 42);
    print_list(head);
    while (head != NULL) {
        Node *next = head->next;
        free(head);
        head = next;
    }
    return 0;
}

🧪 **Experimente:** tire o 1, o último nó. Depois tire todos os nós um a um até a lista ficar vazia.

## 7. Além das listas simples

A mesma ideia monta outras estruturas:

- uma **lista duplamente ligada** acrescenta um ponteiro `prev`, para andar nos dois sentidos e tirar um nó sem procurar o anterior;
- uma **pilha** é uma lista em que você põe e tira na cabeça;
- uma **fila** põe no fim e tira na cabeça;
- uma **árvore binária** dá a cada nó dois ponteiros, `left` e `right`.

In [ ]:
#include <stdio.h>

typedef struct Tree {
    int value;
    struct Tree *left;
    struct Tree *right;
} Tree;

int sum(const Tree *t) {
    if (t == NULL) {
        return 0;
    }
    return t->value + sum(t->left) + sum(t->right);
}

int main(void) {
    Tree leaf1 = {2, NULL, NULL};
    Tree leaf2 = {9, NULL, NULL};
    Tree root = {5, &leaf1, &leaf2};
    printf("soma da árvore: %d\n", sum(&root));
    return 0;
}

🧪 **Experimente:** acrescente um nó com valor 4 embaixo de `leaf1` e confira a nova soma.

## Em resumo

Uma **lista ligada** é uma corrente de nós alocados um por um, cada um com um valor e um ponteiro para o próximo nó; o último aponta para `NULL`. Acrescentar ou remover um nó só muda alguns ponteiros, por mais longa que a lista seja.

```c
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *push(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) exit(1);
    n->value = value;
    n->next = head;
    return n;
}

int main(void) {
    Node *head = NULL;
    for (int i = 1; i <= 3; i++) head = push(head, i);
    for (Node *p = head; p != NULL; p = p->next) printf("%d ", p->value);
    while (head != NULL) {
        Node *next = head->next;
        free(head);
        head = next;
    }
    return 0;
}
```

Com um ponteiro `prev` também, a lista é **duplamente ligada**; com `left` e `right`, os nós formam uma **árvore**.

**Cuidado:** guarde `head->next` antes do `free(head)`: depois do free, o nó não pode ser lido.

## Leituras

- [Lecture 5: Data Structures](https://cs50.harvard.edu/x/2026/notes/5/#data-structures) (CS50x (Harvard), lecture notes, 25 min): De "Data Structures" até "Trees": filas e pilhas, por que arrays são difíceis de aumentar, listas ligadas desenhadas como caixas e setas, e árvores binárias de busca. Tabelas hash e tries vêm depois, para mais adiante.
- [Self-Referential structs](https://beej.us/guide/bgc/html/split/structs-ii-more-fun-with-structs.html#self-referential-structs) (Beej's Guide to C Programming, chapter 20, 4 min): Opcional: por que uma struct pode conter um ponteiro para o próprio tipo (mas não a si mesma), com uma pequena lista ligada montada e percorrida.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Calculadora RPN** · Fácil
- **Fila de impressão** · Médio
- **Desfazer e refazer** · Médio
- **Contador de palavras com ABB** · Difícil
- Teste: **Editor de linhas** · Difícil